# ITSA Pre-alignment of training and testing signals 

In [ ]:
from sklearnex import patch_sklearn

patch_sklearn()

import numpy as np
import scipy.io
import os

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.covariance import LedoitWolf

from pyriemann.estimation import Covariances
from pyriemann.classification import TSclassifier
from pyriemann.spatialfilters import CSP
from pyriemann.tangentspace import TangentSpace
from pyriemann.utils.mean import mean_logeuclid
#from pyriemann.classification import SVC

from sklearn.svm import SVC

import h5py

import time

from sklearn.model_selection import StratifiedKFold

In [ ]:
start_time = time.time()

# Functions

In [ ]:
import os
import h5py
import numpy as np
import scipy.io

# Load testing datasets for a given subject (saved as .mat files) 
def load_testing_data(testing_loading_path, test_subject_id, n_components=108):
    
    testing_dataset_path = os.path.join(testing_loading_path, 'TestingDataset.mat') 
 
    with h5py.File(testing_dataset_path, 'r') as f_testing_dataset:
            testing_dataset = f_testing_dataset['TestingDataset']
            subject_cell_ref = testing_dataset[test_subject_id, 0]
            subject_data = f_testing_dataset[subject_cell_ref]
            X_eval_dataset = np.transpose(subject_data['x'][:], (0, 1, 2))
            Y_test_dataset = np.squeeze(subject_data['y'][:])


    # Organize results into a dictionary
    data = {
        "X_eval_dataset": X_eval_dataset,
        "Y_test_dataset": Y_test_dataset,
    }

    return data


In [ ]:
import os
import h5py
import numpy as np
import scipy.io

# Load training datasets for a given subject (saved as .mat files) 
def load_training_subject_data(training_loading_path, test_subject_id, train_subject_id, n_components=108):

    training_dataset_path = os.path.join(training_loading_path, 'TrainingDataset.mat') 
 
    with h5py.File(training_dataset_path, 'r') as f_training_dataset:
        training_dataset = f_training_dataset['TrainingDataset']
        training_cell_ref = training_dataset[test_subject_id,0]
        training_cell = f_training_dataset[training_cell_ref]

        subject_ref = training_cell[train_subject_id,0]
        subject_data = f_training_dataset[subject_ref]

        X_train_dataset = np.transpose(subject_data['x'][:], (0, 1, 2))
        Y_train_dataset = np.squeeze(subject_data['y'][:])
 

    # Organize results into a dictionary
    data = {
        "X_train_dataset": X_train_dataset,
        "Y_train_dataset": Y_train_dataset,
    }

    return data

In [ ]:
# Recentering function 
import numpy as np
from scipy.linalg import fractional_matrix_power
from pyriemann.utils.tangentspace import tangent_space

def Recentering_TSProjection(C, M):
    """
    1. Centers each covariance matrix: M^(-1/2) C M^(-1/2))
    2. Projects to TS (& vectorises) using identity matrix as 'new centre'
    
    """
    
    Nt, _, _ = C.shape # Number of trials 
    
    M_inv_sqrt = fractional_matrix_power(M, -0.5) # Compute M^(-1/2)
    
    # 1. Recentering step (on each covariance matrix) 
    X_centered = np.array([M_inv_sqrt @ C[i] @ M_inv_sqrt for i in range(Nt)])
    #print(X_centered.shape)
    
    #2. TS projection 
    Centered_M = np.eye(C.shape[1]) # New center after centering is the identity matrix 
    X_TS = tangent_space(X_centered, Centered_M) # Project to TS (& vectorise)

    
    return X_TS

In [ ]:
def Rescaling(C):
    """
    Rescale all tangent vectors: c_tilde = c / (1/N_t * sum_n ||c_n||)

    """
    
    Nt = C.shape[0] # Number of vectors
    
    # Compute the average magnitude (1/N_t * sum(||c_n||))
    avg_magnitude = np.sum(np.linalg.norm(C, axis=1)) / Nt

    # Normalize all vectors simultaneously
    c_tilde  = C / avg_magnitude

    return c_tilde 

In [ ]:
# Rotation Functions

# Mean per class 
def ClassMean(Dataset, Labels):
    """
    Compute the class-wise mean based on the provided equation:
        s̄_k = (1 / N_k) * Σ s̃_i  (for y_i = k)
    
    """
    unique_classes = np.unique(Labels)  # Find unique classes 
    class_means = []

    for cls in unique_classes: # for each class
        # Select data corresponding to the current class
        class_data = Dataset[Labels == cls]  # Filter rows where label == cls
        N_k = class_data.shape[0]  # Number of trials for class k
        
        # Compute mean
        class_mean = np.mean(class_data, axis=0)
        
        # Store result
        class_means.append(class_mean)
    
    C_bar = np.column_stack(class_means)

    return C_bar

# Load subject's dataset

In [ ]:
def load_data(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
        return data

In [ ]:
TransformedTraining = {}
TransformedTesting = {}
indices_split = {}

Test_Ids= list(range(18)) # Testing subject index
N_train = 17


for test_sub_id in Test_Ids:
    print(f"Testing subject {test_sub_id}:")
    test_subject_data = load_testing_data(LoadingPath_Non, test_sub_id, n_components=Nc)


    X_test_dataset = test_subject_data["X_test_dataset"]
    Y_test_dataset = test_subject_data["Y_test_dataset"]
    
    X_test_Block = test_subject_data["X_test_Block"] # For each trial and corresponding label - tracks the adaptive/non-adaptive 'block' it belonged to
    Y_test_Block = test_subject_data["Y_test_Block"]
    
    # Sanity check
    print("Loaded testing datatsets shape: ")
    print(X_eval_dataset.shape)
    
# ---------------------- 1. Recentre (TESTING) - recentres all trials to have centre of mass equal to identity matrix, followed by tangent space mapping (ref = I) ----------------------     
    # Find covariance matrices
    cov_estimator = Covariances(estimator='lwf')
    Test_cov = cov_estimator.transform(X_eval_dataset)

    # Find log-euclid centre ('M')
    Test_M = mean_logeuclid(Test_cov)

    # Centre and project to TS (& vectorise)
    Test_Centered = Recentering_TSProjection(Test_cov, Test_M)


    # ---------------------- 1. Recentre (TRAINING) - recentres all trials to have centre of mass equal to identity matrix, followed by tangent space mapping (ref = I) ----------------------     
    Train_Centered_list = []
    Y_train_list=[]
    
    # Load each (ind) tranining dataset
    for train_sub_id in range(N_train):
        train_subject_data = load_training_subject_data(LoadingPath_Non, test_sub_id, train_sub_id, n_components=Nc)


        X_train_dataset = train_subject_data["X_train_dataset"]
        Y_train_sub_dataset = train_subject_data["Y_train_dataset"]
    

        # Sanity check
        # print("Loaded training datatsets shape: ")
        # print(X_train_dataset.shape)
        
        # Find covariance matrices
        cov_estimator = Covariances(estimator='lwf')
        Train_cov = cov_estimator.transform(X_train_dataset)

        # Find log-euclid centre ('M')
        Train_M = mean_logeuclid(Train_cov)

        # Centre and project to TS (& vectorise)
        Train_sub_Centered = Recentering_TSProjection(Train_cov, Train_M)
        
        # Concatenate each subjects' centred vector and corresponding labels 
        Train_Centered_list.append(Train_sub_Centered)  
        Y_train_list.append(Y_train_sub_dataset)
        
    Train_Centered = np.concatenate(Train_Centered_list, axis=0)
    Y_train_dataset = np.concatenate(Y_train_list, axis = 0)
    
    print(f"Labels sizes: Training {Y_train_dataset.shape}, Testing {Y_test_dataset.shape}")
    
    #Sanity Check
    print("Centred shapes: ")
    print(Train_Centered.shape)
    print(Test_Centered.shape)
    
    # ---------------------- 2. Rescale - match matrix dispersion around mean in both 'source' and 'target' (setting the average norm within set to be 1) ----------------------
    Train_Rescale = Rescaling(Train_Centered)
    Test_Rescale = Rescaling(Test_Centered)

    #Sanity Check
    print("Rescaled shapes: ")
    print(Train_Rescale.shape)
    print(Test_Rescale.shape)
    
    # ---------------------- 3. Rotation (Alignment of 'target' vectors) - Align each mean of each class as much as possible (using Eulidean Procrustes procedure)  ----------------------
    # ---------------------- Once target vectors are aligned, can be used with models trained using 'Train_Rescale' ---------------------- 
    # Calculate anchor points for each class 
    
    
    # Split Testing into calibration and testing 
    Eval_Rotated_list=[] # Intialise storage for evaluation subset after transformation parameters have been applied
    Eval_labels_list=[]
    
    # Store indices split per testing subject and fold
    calibration_idx_list = []
    eval_idx_list = []
    
    indices_split[test_sub_id] = {}
    
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42) # 50:50 split calibration and evaluation
    for fold_idx, (calibration_idx, eval_idx) in enumerate(skf.split(X_test_Block, Y_test_Block)): # Avoid leakage as a result of overlapping window
    # split at block level i.e., all windows from same block are either exclusively in calibration or evaluation subset
    
        print(f"Fold index: {fold_idx}")
        print(f"Indices for calibration subset: {calibration_idx}")
        print(f"Indices for evaluation subset: {eval_idx}")
        
        # Store per testing subject and fold
        indices_split[test_sub_id][f"Fold{fold_idx + 1}"] = {
            "calibration_idx": calibration_idx,
            "eval_idx": eval_idx
        }

        # Assign calibration and evaluation 'block' indices
        X_calibration_block, X_eval_block = X_test_Block[calibration_idx], X_test_Block[eval_idx] 
        Y_calibration_block, Y_eval_block = Y_test_Block[calibration_idx], Y_test_Block[eval_idx]
        
        # For each Test_Rescale trial (& label) extract corresponding 'block' ID and assign to calibration and evaluation assignment 
        X_calibration, Y_calibration  = BlockID(Test_Rescale, Y_test_dataset, X_calibration_block, Y_calibration_block)
        X_eval, Y_eval  = BlockID(Test_Rescale, Y_test_dataset, X_eval_block, Y_eval_block)

        Train_AnchorPoints=ClassMean(Train_Rescale, Y_train_dataset)
        Calib_AnchorPoints=ClassMean(X_calibration, Y_calibration) # Only calibration subset

        # Sanity check
        print("Anchor Points shape: ")
        print(Train_AnchorPoints.shape)
        print(Calib_AnchorPoints.shape)

        # Cross-product matrix 
        c_st = Train_AnchorPoints @ Calib_AnchorPoints.T

        #Sanity check - should be (Nf, Nf)
        print("c_st shape: ")
        print(c_st.shape) 

        # Perform Singular value decomposition on c_st
        U, D, VT = np.linalg.svd(c_st, full_matrices=False)

        # Find number of Nv vectors that explains 99.9% varaince 
        explained_variance = D**2
        total_variance = np.sum(explained_variance)
        cumulative_explained_variance = np.cumsum(explained_variance) / total_variance
        Nv = np.argmax(cumulative_explained_variance >= 0.999) + 1  # +1 because of 0-based indexing

        U_tilde = U[:, :Nv] # Truncate using only Nv vectors
        VT_tilde = VT[:Nv, :]

        # Sanity check 
        print("truncated U and VT: ")
        print(U_tilde.shape)
        print(VT_tilde.shape)

        print(X_calibration.shape)

        Nt = X_eval.shape[0]
        Eval_Rotated = np.zeros_like(X_eval) # Initialise storage

        # Align each evaluation trial
        for t in range(Nt):
            Eval_Rotated[t] = U_tilde @ VT_tilde @ X_eval[t]

        print("Rotated Test: ")
        print(Eval_Rotated.shape)

         # Save for this fold
        Eval_Rotated_list.append(Eval_Rotated)
        Eval_labels_list.append(Y_eval)

        if test_sub_id not in TransformedTesting:
            TransformedTesting[test_sub_id] = {}

        TransformedTesting[test_sub_id][f"Fold{fold_idx + 1}"] = {
            "x": Eval_Rotated,
            "y": Y_eval
        }

    if test_sub_id not in TransformedTraining:
        TransformedTraining[test_sub_id] = {}
        
    # Store per subject (transformed training and testing datasets)
    TransformedTraining[test_sub_id] = {
        "x": Train_Rescale,
        "y": Y_train_dataset
    }


In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time/60, "minutes")

In [ ]:
import os
import pickle

# Define saving paths for the variables
training_saving_path = os.path.join(SavingPath, 'TransformedTraining.pkl')
testing_saving_path = os.path.join(SavingPath, 'TransformedTesting.pkl')

indices_saving_path = os.path.join(SavingPath, 'Indices_Split.pkl')


# Save TransformedTraining
with open(training_saving_path, 'wb') as file:
    pickle.dump(TransformedTraining, file)

# Save TransformedTesting
with open(testing_saving_path, 'wb') as file:
    pickle.dump(TransformedTesting, file)

# Save Indices
with open(indices_saving_path, 'wb') as file:
    pickle.dump(indices_split, file)